# 02 · Feature Engineering (Breakout Signals)

Action: Create transparent proxy features with business meaning.
So what: We quantify momentum into 0–100 scores per artist.


In [ ]:
# Notebook bootstrap: ensure project root is on sys.path for `import src.*`
import os, sys
from pathlib import Path
nb_cwd = Path.cwd()
root = None
for p in [nb_cwd, *nb_cwd.parents]:
    if (p / 'pyproject.toml').exists() or (p / 'src').exists():
        root = p; break
if root and str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ['PYTHONPATH'] = str(root) + (':' + os.environ.get('PYTHONPATH','') if os.environ.get('PYTHONPATH') else '')
print(f'Project root: {root}')


In [ ]:
import pandas as pd
from src.config import load_settings
from src.metrics import assemble_artist_scores, classify_breakout_candidates
from src import qa as _qa

settings = load_settings()

with _qa.time_limit(settings.execution_timeout_s):
    ts = pd.read_csv(settings.time_series_csv, parse_dates=['metrics_date'])

# Guardrails on input
_qa.assert_row_limit(ts, settings.max_rows)
_qa.assert_memory_cap_mb(ts, settings.memory_cap_mb)
_qa.assert_columns(ts, ['artist_name','metrics_date','views_per_day'])

with _qa.time_limit(settings.execution_timeout_s):
    artist_scores = assemble_artist_scores(ts)
    classified = classify_breakout_candidates(artist_scores)

# Persist engineered features for downstream notebooks
out_path = settings.processed_dir / 'artist_breakout_scores.csv'
classified.to_csv(out_path, index=False)
print({'saved': str(out_path), 'rows': len(classified)})
display(classified.sort_values('composite_score', ascending=False).head(10))
